## Importación de librerías


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

palette = sns.color_palette("deep", 3)
sns.set_theme(style="whitegrid", palette=palette)
class_palette = dict(zip([1, 2, 3], palette))
survival_palette = {0: palette[0], 1: palette[1]}
model_palette = dict(zip(["Inicial", "Sin atípicos", "Optimizado sin atípicos"], palette))
heatmap_cmap = sns.light_palette(palette[0], as_cmap=True)
correlation_cmap = sns.blend_palette([palette[1], "white", palette[0]], as_cmap=True)


## Carga de datos


In [ ]:
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(raiz / "data" / "titanic.csv")
df.head()


## Exploración inicial


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


## Valores faltantes


In [ ]:
pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
}).sort_values("nulos", ascending=False)


## Detección de duplicados


In [ ]:
df.duplicated().sum()


## Distribución de la supervivencia


In [ ]:
df["survived"].value_counts(normalize=True) # "normalize" retorona la proporción de cada clase en lugar del conteo


In [ ]:
sns.countplot(data=df, x="survived", hue="survived", palette=survival_palette, saturation=1, legend=False)
plt.show()


## Variables categóricas y supervivencia


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for column, ax in zip(["sex", "pclass", "embarked"], axes):
    sns.countplot(data=df, x=column, hue="survived", palette=survival_palette, saturation=1, ax=ax)
plt.tight_layout()
plt.show()


## Distribución de variables numéricas


In [ ]:
df_numeric = df.select_dtypes(include="number")
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for column, ax in zip(df_numeric.columns, axes.flat):
    sns.histplot(data=df_numeric, x=column, bins=20, color=palette[0], ax=ax)
plt.tight_layout()
plt.show()


## Relaciones entre variables


In [ ]:
sns.pairplot(df_numeric, hue="survived", diag_kind="kde", palette=survival_palette)
plt.show()


## Matriz de correlación


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_numeric.corr(), annot=True, cmap=correlation_cmap, center=0, vmin=-1, vmax=1)
plt.tight_layout()
plt.show()


## Detección de valores atípicos de fare por pclass


In [ ]:
sns.boxplot(data=df, x="pclass", y="fare", hue="pclass", palette=class_palette, saturation=1, legend=False)
plt.show()


In [ ]:
fare_groups = df.groupby("pclass")["fare"]
q1 = fare_groups.transform("quantile", q=0.25)
q3 = fare_groups.transform("quantile", q=0.75)
iqr = q3 - q1
outliers = (df["fare"] < q1 - 1.5 * iqr) | (df["fare"] > q3 + 1.5 * iqr)
outliers.groupby(df["pclass"]).sum().rename("tarifas_atipicas")


## Selección de variables predictoras y objetivo


In [ ]:
numeric_features = ["age", "fare"]
categorical_features = ["sex", "embarked", "pclass", "sibsp", "parch"]

X = df[numeric_features + categorical_features]
y = df["survived"]


## División en entrenamiento y prueba


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


## Eliminación de tarifas atípicas por pclass: IQR de entrenamiento


In [ ]:
def fare_inliers(data):
    fare = data.groupby("pclass")["fare"]
    q1 = fare.transform("quantile", q=0.25)
    q3 = fare.transform("quantile", q=0.75)
    iqr = q3 - q1
    return data["fare"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr) | data["fare"].isna()

keep = fare_inliers(X_train)
X_train_clean = X_train.loc[keep].copy()
y_train_clean = y_train.loc[keep].copy()

print(f"Antes: {len(X_train)} | Después: {len(X_train_clean)} | Eliminados: {(~keep).sum()}")


## Fare antes y después de eliminar atípicos de entrenamiento


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=False, sharey=False)
for row, (title, data) in enumerate([("Antes", X_train), ("Después", X_train_clean)]):
    sns.boxplot(
        data=data, x="pclass", y="fare", hue="pclass", palette=class_palette,
        saturation=1, legend=False, ax=axes[row, 0]
    )
    sns.kdeplot(
        data=data, x="fare", hue="pclass", palette=class_palette,
        fill=True, common_norm=False, alpha=0.25, cut=0, ax=axes[row, 1]
    )
    axes[row, 0].set_title(f"{title}: tarifas por clase")
    axes[row, 0].set_xlabel("Clase")
    axes[row, 0].set_ylabel("Fare")
    axes[row, 1].set_title(f"{title}: distribución de tarifas")
    axes[row, 1].set_xlabel("Fare")
    axes[row, 1].set_ylabel("Densidad")
plt.tight_layout()
plt.show()


## Imputación y codificación de variables


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## Árbol inicial con y sin eliminación de atípicos


In [ ]:
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("tree", DecisionTreeClassifier(random_state=42))
])
baseline_model.fit(X_train, y_train) # modelo control - datos originales con outliers
clean_model = clone(baseline_model).fit(X_train_clean, y_train_clean) # modelo sin atípicos


## Modelo optimizado


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X_train, y_train))
clean_folds = []

for train_idx, valid_idx in folds:
    keep_fold = fare_inliers(X_train.iloc[train_idx]).to_numpy()
    clean_folds.append((train_idx[keep_fold], valid_idx))


## GridSearchCV: selección por AUC con 5 folds


In [ ]:
param_grid = {
    "tree__criterion": ["gini", "entropy"],
    "tree__max_depth": [3, 4, 5, 6],
    "tree__min_samples_split": [2, 10],
    "tree__min_samples_leaf": [1, 3, 5]
}

scoring = {"accuracy": "accuracy", "auc": "roc_auc"}
grid_search = GridSearchCV(
    baseline_model, param_grid, cv=clean_folds, scoring=scoring,
    refit=False, error_score="raise"
)
grid_search.fit(X_train, y_train)


In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
best_index = results["mean_test_auc"].idxmax()
best_params = results.loc[best_index, "params"]
model = clone(baseline_model).set_params(**best_params)
model.fit(X_train_clean, y_train_clean)
best_params


In [ ]:
results[[
    "mean_test_auc", "std_test_auc", "mean_test_accuracy", "params"
]].sort_values("mean_test_auc", ascending=False).head()


## AUC y accuracy por fold: el optimizado usa los folds de selección del grid


In [ ]:
fold_results = []
for name, splits in [("Inicial", folds), ("Sin atípicos", clean_folds)]:
    scores = cross_validate(
        baseline_model, X_train, y_train, cv=splits,
        scoring=scoring, error_score="raise"
    )
    for fold in range(5):
        fold_results.append({
            "modelo": name, "fold": fold + 1,
            "auc": scores["test_auc"][fold],
            "accuracy": scores["test_accuracy"][fold]
        })

for fold in range(5):
    fold_results.append({
        "modelo": "Optimizado sin atípicos", "fold": fold + 1,
        "auc": results.loc[best_index, f"split{fold}_test_auc"],
        "accuracy": results.loc[best_index, f"split{fold}_test_accuracy"]
    })

cv_results = pd.DataFrame(fold_results)
cv_pivot = cv_results.pivot(index="fold", columns="modelo", values=["auc", "accuracy"])
cv_pivot


In [ ]:
comparison_cv = cv_results.groupby("modelo")[["auc", "accuracy"]].agg(["mean", "std"])
comparison_cv


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for metric, ax in zip(["auc", "accuracy"], axes):
    sns.barplot(
        data=cv_results, x=metric, y="modelo", hue="modelo", palette=model_palette, saturation=1,
        errorbar="sd", capsize=0.15, legend=False, ax=ax
    )
    ax.set_xlim(0, 1)
    ax.set_xlabel(metric.upper())
    ax.set_ylabel("")
    ax.set_title(f"{metric.upper()}: media y desviación estándar")
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.3f", label_type="center", color="white", fontsize=11)
plt.tight_layout()
plt.show()


## Curvas de aprendizaje del árbol seleccionado: accuracy y AUC


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
learning_results = {}

for metric, ax in zip(["accuracy", "roc_auc"], axes):
    sizes, train_scores, valid_scores = learning_curve(
        model, X_train, y_train, cv=clean_folds,
        train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0],
        scoring=metric, shuffle=True, random_state=42, error_score="raise"
    )
    learning_results[metric] = (sizes, train_scores, valid_scores)
    curves = []
    for label, scores in [("Entrenamiento", train_scores), ("Validación", valid_scores)]:
        values = pd.DataFrame(scores, index=sizes).rename_axis("pasajeros").reset_index()
        values = values.melt(id_vars="pasajeros", var_name="fold", value_name="puntuación")
        values["conjunto"] = label
        curves.append(values)
    sns.lineplot(
        data=pd.concat(curves), x="pasajeros", y="puntuación", hue="conjunto",
        palette={"Entrenamiento": palette[0], "Validación": palette[1]},
        errorbar="sd", marker="o", ax=ax
    )
    ax.set_xlabel("Pasajeros de entrenamiento tras el filtro del fold")
    ax.set_ylabel(metric)
    ax.set_ylim(0.4, 1.02)

plt.tight_layout()
plt.show()


## Evaluación final en el conjunto de prueba


In [ ]:
models = {
    "Inicial": baseline_model,
    "Sin atípicos": clean_model,
    "Optimizado sin atípicos": model
}

test_results = []
for name, estimator in models.items():
    predictions = estimator.predict(X_test)
    probabilities = estimator.predict_proba(X_test)[:, 1]
    test_results.append({
        "modelo": name,
        "accuracy_prueba": accuracy_score(y_test, predictions),
        "auc_prueba": roc_auc_score(y_test, probabilities)
    })

comparison_test = pd.DataFrame(test_results).set_index("modelo")
y_pred = model.predict(X_test)
comparison_test


## Comparación de curvas ROC en prueba sin eliminar pasajeros


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, estimator in models.items():
    probabilities = estimator.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    sns.lineplot(
        x=fpr, y=tpr, estimator=None, sort=False, errorbar=None,
        label=f"{name} (AUC={auc:.3f})", color=model_palette[name], ax=ax
    )
sns.lineplot(x=[0, 1], y=[0, 1], color="gray", linestyle="--", label="Azar", ax=ax)
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curvas ROC en prueba")
plt.tight_layout()
plt.show()


In [ ]:
sns.heatmap(
    confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap=heatmap_cmap, cbar=False,
    xticklabels=["No sobrevivió", "Sobrevivió"],
    yticklabels=["No sobrevivió", "Sobrevivió"]
)
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.title("Matriz de confusión del árbol optimizado")
plt.tight_layout()
plt.show()


## Importancia de las variables


In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = pd.Series(model.named_steps["tree"].feature_importances_, index=feature_names)
importance_table = importances.rename_axis("variable").reset_index(name="importancia")

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_table.sort_values("importancia", ascending=False),
    x="importancia", y="variable", color=palette[0], saturation=1
)
plt.xlabel("Importancia")
plt.ylabel("")
plt.tight_layout()
plt.show()


## Modelo reducido con las cinco características seleccionadas


In [ ]:
selected_numeric = ["age", "fare"]
selected_binary = ["sex_female", "pclass_3", "embarked_S"]

X_selected = X[selected_numeric].copy()
X_selected["sex_female"] = X["sex"].map({"female": 1, "male": 0})
X_selected["pclass_3"] = X["pclass"].eq(3).astype(float).where(X["pclass"].notna())
X_selected["embarked_S"] = X["embarked"].eq("S").astype(float).where(X["embarked"].notna())

X_train_selected = X_selected.loc[X_train.index]
X_test_selected = X_selected.loc[X_test.index]
X_train_selected_clean = X_selected.loc[X_train_clean.index]
X_train_selected.head()


In [ ]:
selected_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), selected_numeric),
    ("binary", SimpleImputer(strategy="most_frequent"), selected_binary)
])

selected_pipeline = Pipeline([
    ("preprocessor", selected_preprocessor),
    ("tree", DecisionTreeClassifier(random_state=42))
])


## Optimización del modelo reducido: mismo grid y mismos 5 folds


In [ ]:
selected_search = GridSearchCV(
    selected_pipeline, param_grid, cv=clean_folds, scoring=scoring,
    refit=False, error_score="raise"
)
selected_search.fit(X_train_selected, y_train)
selected_results = pd.DataFrame(selected_search.cv_results_)
selected_best_index = selected_results["mean_test_auc"].idxmax()
selected_best_params = selected_results.loc[selected_best_index, "params"]

selected_model = clone(selected_pipeline).set_params(**selected_best_params)
selected_model.fit(X_train_selected_clean, y_train_clean)
selected_best_params


## Reportes de clasificación y AUC: todas frente a seleccionadas


In [ ]:
feature_models = {
    "Todas": (model, X_test),
    "Seleccionadas": (selected_model, X_test_selected)
}

for name, (estimator, test_data) in feature_models.items():
    predictions = estimator.predict(test_data)
    probabilities = estimator.predict_proba(test_data)[:, 1]
    print(name)
    print(classification_report(
        y_test, predictions,
        target_names=["No sobrevivió", "Sobrevivió"], zero_division=0
    ))
    print(f"AUC: {roc_auc_score(y_test, probabilities):.4f}\n")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, (estimator, test_data) in feature_models.items():
    probabilities = estimator.predict_proba(test_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    sns.lineplot(
        x=fpr, y=tpr, estimator=None, sort=False, errorbar=None,
        label=f"{name} (AUC={auc:.3f})",
        color=palette[2] if name == "Todas" else palette[1],
        linestyle="--" if name == "Seleccionadas" else "-", ax=ax
    )
sns.lineplot(x=[0, 1], y=[0, 1], color="gray", linestyle="--", label="Azar", ax=ax)
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("ROC: todas frente a seleccionadas")
plt.tight_layout()
plt.show()


## Visualización del árbol optimizado con todas las características


In [ ]:
plt.figure(figsize=(24, 12))
nodes = plot_tree(
    model.named_steps["tree"],
    feature_names=feature_names,
    class_names=["No sobrevivió", "Sobrevivió"],
    max_depth=3,
    filled=False,
    rounded=True,
    fontsize=9
)
for node in nodes:
    color = palette[1] if "class = Sobrevivió" in node.get_text() else palette[0]
    if node.get_bbox_patch() is not None:
        node.get_bbox_patch().set_facecolor(sns.light_palette(color, n_colors=3)[1])
plt.title("Árbol optimizado: primeros tres niveles de profundidad")
plt.show()
